In [1]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook import (
    check_value,  # noqa: F401
    display_table,
    display_text,
    set_params_columns,
    sig_figs,  # noqa: F401
)
from structural_tools.notebook import feet_inches as fi
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_floor = 35 / 1000 * ksf
U_pavers = 15 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
# Plan dimensions
l_C1__C2 = fi(31, 4)
l_C2__C3 = fi(31, 5)
l_C3__C4 = fi(26, 5)
l_C4__C5 = fi(17, 11)
#
l_C6__C7 = fi(21, 4)
l_C7__C8 = fi(26, 6)
l_C8__C9 = fi(36, 5, 1 / 2)

l_C1__C21 = fi(35, 1, 1 / 2)
l_C21__C29 = fi(24, 8)
l_C29__C42 = fi(34, 4)
l_C42__C55 = fi(24, 8)
#
l_C59__C7 = fi(21, 4)
l_C7__C83 = fi(36, 10)
l_C83__C9 = fi(26, 1, 1 / 2)

l_C3__C33 = fi(8, 10)
l_C33__C4 = l_C3__C4 - l_C3__C33
l_C7__C73 = fi(7, 7, 1 / 4)
l_C6__C73 = l_C6__C7 + l_C7__C73
l_C73__C8 = l_C7__C8 - l_C7__C73
l_C8__C81 = fi(1, 5, 0.75)
l_C81__C83 = fi(8, 10)
l_C7__C81 = l_C7__C8 + l_C8__C81

l_CA__CB = fi(30, 9)
l_CB__CC = fi(5, 6)
l_CC__CD = fi(30, 9)

l_CW__CX = fi(30, 9)
l_CX__CY = fi(5, 6)
l_CY__CZ = fi(30, 9)

l_CW__CW6 = fi(19, 11, 0.25)
l_CW6__CX = l_CW__CX - l_CW__CW6
l_CY__CY6 = fi(19, 8, 0.25)
l_CY6__CZ = l_CY__CZ - l_CY__CY6

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 13.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10.5 * ft
h_wall__5 = 10.5 * ft

<IPython.core.display.Latex object>

In [7]:
P_wall = (
    7
    + fi(30, 9)
    + fi(79, 9, 1 / 2)
    + fi(86, 1, 1 / 4)
    + fi(30, 9)
    + fi(31, 4)
    + fi(36, 3)
    + fi(131, 3 / 4)
    + fi(100, 4, 3 / 4)
    + fi(36, 3)
) * ft

In [8]:
%%render
P_wall

<IPython.core.display.Latex object>

The areas of the building are:

In [9]:
%%render params
A_level = 13417 * ft**2
A_pavers = 907 * ft**2
A_floor = A_level - A_pavers

<IPython.core.display.Latex object>

In [10]:
%%render
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall
A_wall__5 = h_wall__5 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [11]:
%%render
W_topfloor = U_floor * A_floor
W_pavers = (U_floor + U_pavers) * A_pavers
W_floor__typ = U_floor * A_level

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4. The 5th (top) level has pavers in the N-E corner.

In [12]:
set_params_columns(4)

In [13]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_floor__5 = W_topfloor + W_pavers

<IPython.core.display.Latex object>

The roof weight is

In [14]:
%%render
W_roof = U_roof * A_floor

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [15]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__5 = U_wall * (A_wall__5 / 2 + A_wall__4 / 2)
W_wall__roof = U_wall * (A_wall__5 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [16]:
%%render 
W_level__2 = W_floor__2 + W_wall__2
W_level__3 = W_floor__3 + W_wall__3
W_level__4 = W_floor__4 + W_wall__4
W_level__5 = W_floor__5 + W_wall__5
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [17]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__5 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [18]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [19]:
set_params_columns(5)

In [20]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [21]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [22]:
set_params_columns(1)

In [23]:
%%render params
h_top__roof = fi(71, 3) * ft
h_bottom__roof = fi(58, 7) * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [24]:
%%render
h_n__check = check_value(h_n, 65 * ft, "<=")

<IPython.core.display.Latex object>

We will approximate the building period.

In [25]:
h_n = float(h_n)  # to prevent weirdness with units

In [26]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [27]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [28]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [29]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [30]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [31]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [32]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [33]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [34]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [35]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Ground", "Level 2", "Level 3", "Level 4", "Level 5", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            float(h_wall__5),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__5),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [36]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [37]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,348.640312,55.5,19349.537344,0.231894,61.317250,0.000000,0.000000
Level 5,10.5,554.980625,45.0,24974.128125,0.299302,79.141161,61.317250,643.831124
Level 4,10.5,541.375625,34.5,18677.459063,0.223840,59.187484,140.458411,2118.644441
Level 3,10.5,541.375625,24.0,12993.015000,0.155715,41.173902,199.645895,4214.926336
Level 2,10.5,551.630000,13.5,7447.005000,0.089249,23.599007,240.819796,6743.534199
Ground,13.5,0.000000,0.0,0.000000,0.000000,0.000000,264.418803,10313.188045


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [38]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Roof", "Level 5", "Level 4", "Level 3", "Level 2"]
walls = {
    # N-S
    "C-2.1": "N-S",
    "C-2.9": "N-S",
    "C-3": "N-S",
    "C-3.3": "N-S",
    "C-4": "N-S",
    "C-4.2": "N-S",
    "C-5": "N-S",
    "C-5.5": "N-S",
    "C-5.9": "N-S",
    "C-6": "N-S",
    "C-7": "N-S",
    "C-7.3": "N-S",
    "C-8--C-W": "N-S",
    "C-8--C-W.6": "N-S",
    "C-8.1": "N-S",
    "C-8.3": "N-S",
    # E-W
    "C-B--C-2": "E-W",
    "C-B--C-4": "E-W",
    "C-B--C-4.9": "E-W",
    "C-W.6--C-7.3": "E-W",
    "C-X--C-5.8": "E-W",
    "C-X--C-6.5": "E-W",
    "C-C--C-2.5": "E-W",
    "C-C--C-3.6": "E-W",
    "C-C--C-5.2": "E-W",
    "C-Y--C-5.8": "E-W",
    "C-Y--C-6.5": "E-W",
    "C-Y--C-7.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (paver weight vs. floor weight, for instance), but the vast majority of the area of each level is typical floor weight.

Due to this building's geometry, we must include an additional 

$$
f_{p,x} = F_x / A_{level}
$$

In [39]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__5 = seismic_loads.loc["Level 5", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [40]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [41]:
d_trib_ns = 67 / 2 * ft

In [42]:
# Wall tributary widths and lengths

t_w__C21 = l_C1__C21 + (l_C21__C29 / 2)
t_w__C29 = (l_C21__C29 / 2) + (l_C29__C42 / 2)
t_w__C42 = (l_C29__C42 / 2) + (l_C42__C55 / 2)
t_w__C55 = (l_C42__C55 / 2) + 0  # placeholder for angled area that will be manualled added

t_w__C3 = l_C2__C3 + (l_C3__C33 / 2)
t_w__C33 = (l_C3__C33 / 2) + (l_C33__C4 / 2)
t_w__C4 = (l_C33__C4 / 2) + (l_C4__C5 / 2)
t_w__C5 = (l_C4__C5 / 2) + 0  # placeholder for angled area that will be manualled added

t_w__C59 = (l_C59__C7 / 2) + 0  # placeholder for angled area that will be manualled added
t_w__C7 = (l_C59__C7 / 2) + (l_C7__C81 / 2)
t_w__C81 = (l_C7__C81 / 2) + (l_C81__C83 / 2)
t_w__C83 = (l_C81__C83 / 2) + l_C83__C9

t_w__C6 = (l_C6__C73 / 2) + 0  # placeholder for angled area that will be manualled added
t_w__C73 = (l_C6__C73 / 2) + (l_C73__C8 / 2)
t_w__C8 = (l_C73__C8 / 2) + l_C8__C9 - 7  # 7 foot jog

t_l__C21 = float(d_trib_ns)
t_l__C29 = float(d_trib_ns)
t_l__C3 = float(d_trib_ns)
t_l__C33 = float(d_trib_ns)
t_l__C4 = float(d_trib_ns)
t_l__C42 = float(d_trib_ns)
t_l__C5 = float(d_trib_ns)
t_l__C55 = float(d_trib_ns)
t_l__C59 = float(d_trib_ns)
t_l__C6 = float(d_trib_ns)
t_l__C7 = float(d_trib_ns)
t_l__C73 = float(d_trib_ns)
t_l__C8__1 = fi(15, 8)
t_l__C8__2 = fi(17, 10)
t_l__C81 = float(d_trib_ns)
t_l__C83 = float(d_trib_ns)

In [43]:
flexible_ns.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 5", "tributary weight"] = float(f_p__5)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__2)

# Nonparallel irregularity
angle_degrees = 23
angle = math.radians(angle_degrees)
angled_walls = ["C-2.1", "C-2.9", "C-3", "C-3.3", "C-4", "C-4.2", "C-5", "C-5.5"]
wall_indices = flexible_ns.index.get_level_values("Wall").isin(angled_walls)
flexible_ns.loc[wall_indices, "tributary weight"] *= math.cos(angle) + 0.3 * math.sin(angle)

# Shear wall height is for the floor below
flexible_ns.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ns.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
l_typ = 29
l_stairwall = 19
l_stairwall_short = fi(14, 3)
l_elevator = 11
wall_tributary = {
    "C-2.1": {"trib_width": float(t_w__C21), "trib_depth": t_l__C21, "wall_length": l_typ},
    "C-2.9": {"trib_width": float(t_w__C29), "trib_depth": t_l__C29, "wall_length": l_typ},
    "C-3": {"trib_width": float(t_w__C3), "trib_depth": t_l__C3, "wall_length": l_stairwall_short},
    "C-3.3": {"trib_width": float(t_w__C33), "trib_depth": t_l__C33, "wall_length": l_stairwall},
    "C-4": {"trib_width": float(t_w__C4), "trib_depth": t_l__C4, "wall_length": l_typ},
    "C-4.2": {"trib_width": float(t_w__C42), "trib_depth": t_l__C42, "wall_length": l_typ},
    "C-5": {"trib_width": float(t_w__C5), "trib_depth": t_l__C5, "wall_length": l_typ},
    "C-5.5": {"trib_width": float(t_w__C55), "trib_depth": t_l__C55, "wall_length": l_typ},
    "C-5.9": {"trib_width": float(t_w__C59), "trib_depth": t_l__C59, "wall_length": l_typ},
    "C-6": {"trib_width": float(t_w__C6), "trib_depth": t_l__C6, "wall_length": l_typ},
    "C-7": {"trib_width": float(t_w__C7), "trib_depth": t_l__C7, "wall_length": l_typ},
    "C-7.3": {"trib_width": float(t_w__C73), "trib_depth": t_l__C73, "wall_length": l_elevator},
    "C-8--C-W": {"trib_width": float(t_w__C8), "trib_depth": t_l__C8__1, "wall_length": fi(9, 6)},
    "C-8--C-W.6": {"trib_width": float(t_w__C8), "trib_depth": t_l__C8__2, "wall_length": l_elevator},
    "C-8.1": {"trib_width": float(t_w__C81), "trib_depth": t_l__C81, "wall_length": l_stairwall_short},
    "C-8.3": {"trib_width": float(t_w__C83), "trib_depth": t_l__C83, "wall_length": l_stairwall},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-2.1", "tributary area"] += 116
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5", "tributary area"] += 447
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5.5", "tributary area"] += 290
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5.9", "tributary area"] += 416
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-6", "tributary area"] += 195
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-8.3", "tributary area"] += 23.5

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [44]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [45]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [46]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [47]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [48]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [49]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [50]:
display_table(
    dataframe=flexible_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Roof",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 5",
    label="tab:shear_wall_forces_level5_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,4.742520,1705.854167,8090.047854,8090.047854,29.00,278.967167,195.277017,310.714286,1,6.0
C-2.9,4.742520,988.250000,4686.795594,4686.795594,29.00,161.613641,113.129549,310.714286,1,6.0
C-3,4.742520,1200.416667,5693.000297,5693.000297,14.25,399.508793,279.656155,310.714286,1,6.0
C-3.3,4.742520,442.479167,2098.466389,2098.466389,19.00,110.445599,77.311920,310.714286,1,6.0
C-4,4.742520,594.625000,2820.021078,2820.021078,29.00,97.242106,68.069474,310.714286,1,6.0
C-4.2,4.742520,988.250000,4686.795594,4686.795594,29.00,161.613641,113.129549,310.714286,1,6.0
C-5,4.742520,747.104167,3543.156607,3543.156607,29.00,122.177814,85.524470,310.714286,1,6.0
C-5.5,4.742520,703.166667,3334.782125,3334.782125,29.00,114.992487,80.494741,310.714286,1,6.0
C-5.9,4.570116,773.333333,3534.223243,3534.223243,29.00,121.869767,85.308837,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,6.121092,1705.854167,10441.691088,18531.738943,29.00,639.025481,447.317837,460.714286,1,4.0
C-2.9,6.121092,988.250000,6049.169630,10735.965224,29.00,370.205697,259.143988,310.714286,1,6.0
C-3,6.121092,1200.416667,7347.861415,13040.861712,14.25,915.148190,640.603733,769.642857,1,2.0
C-3.3,6.121092,442.479167,2708.455894,4806.922282,19.00,252.995910,177.097137,310.714286,1,6.0
C-4,6.121092,594.625000,3639.754608,6459.775685,29.00,222.750886,155.925620,310.714286,1,6.0
C-4.2,6.121092,988.250000,6049.169630,10735.965224,29.00,370.205697,259.143988,310.714286,1,6.0
C-5,6.121092,747.104167,4573.093686,8116.250293,29.00,279.870700,195.909490,310.714286,1,6.0
C-5.5,6.121092,703.166667,4304.148186,7638.930311,29.00,263.411390,184.387973,310.714286,1,6.0
C-5.9,5.898574,773.333333,4561.563542,8095.786785,29.00,279.165062,195.415543,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,4.577796,1705.854167,7809.051706,26340.790649,29.00,908.303126,635.812188,769.642857,1,2.0
C-2.9,4.577796,988.250000,4524.006506,15259.971730,29.00,526.205922,368.344145,460.714286,1,4.0
C-3,4.577796,1200.416667,5495.262140,18536.123853,14.25,1300.780621,910.546435,921.428571,2,4.0
C-3.3,4.577796,442.479167,2025.579184,6832.501467,19.00,359.605340,251.723738,310.714286,1,6.0
C-4,4.577796,594.625000,2722.071711,9181.847397,29.00,316.615427,221.630799,310.714286,1,6.0
C-4.2,4.577796,988.250000,4524.006506,15259.971730,29.00,526.205922,368.344145,460.714286,1,4.0
C-5,4.577796,747.104167,3420.090170,11536.340463,29.00,397.804844,278.463390,310.714286,1,6.0
C-5.5,4.577796,703.166667,3218.953276,10857.883587,29.00,374.409779,262.086845,310.714286,1,6.0
C-5.9,4.411380,773.333333,3411.467095,11507.253880,29.00,396.801858,277.761301,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,3.184553,1705.854167,5432.383796,31773.174444,29.00,1095.626705,766.938693,769.642857,1,2.0
C-2.9,3.184553,988.250000,3147.134961,18407.106691,29.00,634.727817,444.309472,460.714286,1,4.0
C-3,3.184553,1200.416667,3822.791054,22358.914907,14.25,1569.046660,1098.332662,1200.000000,2,3.0
C-3.3,3.184553,442.479167,1409.098563,8241.600030,19.00,433.768423,303.637896,310.714286,1,6.0
C-4,3.184553,594.625000,1893.615104,11075.462500,29.00,381.912500,267.338750,310.714286,1,6.0
C-4.2,3.184553,988.250000,3147.134961,18407.106691,29.00,634.727817,444.309472,460.714286,1,4.0
C-5,3.184553,747.104167,2379.193162,13915.533625,29.00,479.845987,335.892191,460.714286,1,4.0
C-5.5,3.184553,703.166667,2239.271844,13097.155431,29.00,451.626049,316.138235,460.714286,1,4.0
C-5.9,3.068786,773.333333,2373.194501,13880.448380,29.00,478.636151,335.045306,460.714286,1,4.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,1.825241,1705.854167,3113.595212,34886.769656,29.00,1202.992057,842.094440,921.428571,2,4.0
C-2.9,1.825241,988.250000,1803.794561,20210.901252,29.00,696.927629,487.849341,600.000000,1,3.0
C-3,1.825241,1200.416667,2191.049891,24549.964797,14.25,1722.804547,1205.963183,1539.285714,2,2.0
C-3.3,1.825241,442.479167,807.631181,9049.231210,19.00,476.275327,333.392729,460.714286,1,4.0
C-4,1.825241,594.625000,1085.334016,12160.796516,29.00,419.337811,293.536468,310.714286,1,6.0
C-4.2,1.825241,988.250000,1803.794561,20210.901252,29.00,696.927629,487.849341,600.000000,1,3.0
C-5,1.825241,747.104167,1363.645264,15279.178889,29.00,526.868238,368.807766,460.714286,1,4.0
C-5.5,1.825241,703.166667,1283.448731,14380.604162,29.00,495.882902,347.118032,460.714286,1,4.0
C-5.9,1.758888,773.333333,1360.207104,15240.655485,29.00,525.539844,367.877891,460.714286,1,4.0


#### East-West Direction

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

In [51]:
# Wall tributary widths and lengths
t_w__CB = l_CA__CB + l_CB__CC / 2
t_w__CC = l_CB__CC / 2 + l_CC__CD
t_w__CX = l_CW__CX + l_CX__CY / 2
t_w__CY = l_CX__CY / 2 + l_CY__CZ
t_w__CW6 = t_w__CX

t_l__CB__1 = fi(41, 5)
# t_l__CB__2 = fi(23, 5)
t_l__CB__3 = fi(28, 4)
t_l__CB__4 = fi(23, 1)
t_l__CC__1 = fi(77, 4)
t_l__CC__2 = fi(31, 7)
t_l__CC__3 = fi(17, 6)

t_l__CX__1 = fi(13, 10)
t_l__CX__2 = fi(21)
t_l__CW6__1 = fi(49, 1)
t_l__CY__1 = fi(14, 1)
t_l__CY__2 = fi(16, 4)
t_l__CY__3 = fi(59, 9)

In [ ]:
# Wall segment lengths
l_CB__1 = fi(23, 3)
# l_CB__2 = fi(12, 9)
l_CB__3 = fi(9, 11)
l_CB__4 = fi(18, 8)
l_CC__1 = fi(28)
l_CC__2 = fi(19, 4)
l_CC__3 = fi(12, 1)

l_CX__1 = fi(9, 10)
l_CX__2 = fi(18, 11)
l_CW6__1 = fi(19, 1)
l_CY__1 = fi(8, 10)
l_CY__2 = fi(11, 4)
l_CY__3 = fi(19, 7)

In [54]:
flexible_ew.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 5", "tributary weight"] = float(f_p__5)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__2)

# Nonparallel irregularity
angle_degrees = 23
angle = math.radians(angle_degrees)
angled_walls = ["C-B--C-2", "C-B--C-3.3", "C-B--C-4", "C-B--C-4.9", "C-C--C-2.5", "C-C--C-3.6", "C-C--C-5.2"]
wall_indices = flexible_ew.index.get_level_values("Wall").isin(angled_walls)
flexible_ew.loc[wall_indices, "tributary weight"] *= math.cos(angle) + 0.3 * math.sin(angle)

# Shear wall height is for the floor below
flexible_ew.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ew.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "C-B--C-2": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__1, "wall_length": l_CB__1},
    "C-B--C-4": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__3, "wall_length": l_CB__3},
    "C-B--C-4.9": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__4, "wall_length": l_CB__4},
    "C-C--C-2.5": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__1, "wall_length": l_CC__1},
    "C-C--C-3.6": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__2, "wall_length": l_CC__2},
    "C-C--C-5.2": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__3, "wall_length": l_CC__3},
    "C-X--C-5.8": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__1, "wall_length": l_CX__1},
    "C-X--C-6.5": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__2, "wall_length": l_CX__2},
    "C-W.6--C-7.3": {"trib_width": float(t_w__CW6), "trib_depth": t_l__CW6__1, "wall_length": l_CW6__1},
    "C-Y--C-5.8": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__1, "wall_length": l_CY__1},
    "C-Y--C-6.5": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__2, "wall_length": l_CY__2},
    "C-Y--C-7.2": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__3, "wall_length": l_CY__3},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-B--C-2", "tributary area"] += 100
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-B--C-4.9", "tributary area"] -= 120
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-C--C-5.2", "tributary area"] += 160
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-X--C-5.8", "tributary area"] -= 80.25
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-Y--C-5.8", "tributary area"] += 145
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-W.6--C-7.3", "tributary area"] += 35.5

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]
flexible_ew["aspect ratio factor"] = np.where(
    flexible_ew["aspect ratio"] > 2, 1.25 - 0.125 * flexible_ew["wall height"] / flexible_ew["wall length"], 1
)

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [ ]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<=")

<IPython.core.display.Latex object>

In [ ]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [ ]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [ ]:
# Update dataframe
flexible.update(flexible_ew)

In [ ]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [ ]:
display_table(
    dataframe=flexible_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Roof",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 5",
    label="tab:shear_wall_forces_level5_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,4.742520,1146.875000,5284.969667,5284.969667,23.250000,227.310523,159.117366,310.714286,1,6.0
C-B--C-3.3,4.742520,787.250000,2898.209172,2898.209172,12.750000,227.310523,159.117366,310.714286,1,6.0
C-B--C-4,4.742520,508.083333,2254.162690,2254.162690,9.916667,227.310523,159.117366,310.714286,1,6.0
C-B--C-4.9,4.742520,653.291667,4243.129769,4243.129769,18.666667,227.310523,159.117366,310.714286,1,6.0
C-W.6--C-7.3,4.570116,1674.291667,7651.707574,7651.707574,19.083333,400.962842,280.673990,310.714286,1,6.0
C-X--C-5.8,4.570116,383.166667,1698.581956,1698.581956,9.833333,172.737148,120.916004,310.714286,1,6.0
C-X--C-6.5,4.570116,703.500000,3267.611050,3267.611050,18.916667,172.737148,120.916004,310.714286,1,6.0
C-C--C-2.5,4.742520,2590.666667,8809.192508,8809.192508,28.000000,314.614018,220.229813,310.714286,1,6.0
C-C--C-3.6,4.742520,1058.041667,8232.400141,8232.400141,26.166667,314.614018,220.229813,310.714286,1,6.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,6.121092,1146.875000,6821.223023,12106.192690,23.250000,520.696460,364.487522,460.714286,1,4.0
C-B--C-3.3,6.121092,787.250000,3740.670690,6638.879862,12.750000,520.696460,364.487522,460.714286,1,4.0
C-B--C-4,6.121092,508.083333,2909.410537,5163.573226,9.916667,520.696460,364.487522,460.714286,1,4.0
C-B--C-4.9,6.121092,653.291667,5476.537481,9719.667249,18.666667,520.696460,364.487522,460.714286,1,4.0
C-W.6--C-7.3,5.898574,1674.291667,9875.932532,17527.640107,19.083333,918.478958,642.935270,769.642857,1,2.0
C-X--C-5.8,5.898574,383.166667,2192.331663,3890.913619,9.833333,395.686131,276.980292,310.714286,1,6.0
C-X--C-6.5,5.898574,703.500000,4217.451589,7485.062640,18.916667,395.686131,276.980292,310.714286,1,6.0
C-C--C-2.5,6.121092,2590.666667,11369.879210,20179.071717,28.000000,720.681133,504.476793,600.000000,1,3.0
C-C--C-3.6,6.121092,1058.041667,10625.422833,18857.822974,26.166667,720.681133,504.476793,600.000000,1,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,4.577796,1146.875000,5101.403866,17207.596556,23.250000,740.111680,518.078176,600.000000,1,3.0
C-B--C-3.3,4.577796,787.250000,2797.544055,9436.423918,12.750000,740.111680,518.078176,600.000000,1,3.0
C-B--C-4,4.577796,508.083333,2175.867599,7339.440825,9.916667,740.111680,518.078176,600.000000,1,3.0
C-B--C-4.9,4.577796,653.291667,4095.750774,13815.418023,18.666667,740.111680,518.078176,600.000000,1,3.0
C-W.6--C-7.3,4.411380,1674.291667,7385.936544,24913.576651,19.083333,1305.514934,913.860454,921.428571,2,4.0
C-X--C-5.8,4.411380,383.166667,1639.584160,5530.497779,9.833333,562.423503,393.696452,460.714286,1,4.0
C-X--C-6.5,4.411380,703.500000,3154.115291,10639.177931,18.916667,562.423503,393.696452,460.714286,1,4.0
C-C--C-2.5,4.577796,2590.666667,8503.217907,28682.289624,28.000000,1024.367487,717.057241,769.642857,1,2.0
C-C--C-3.6,4.577796,1058.041667,7946.459592,26804.282566,26.166667,1024.367487,717.057241,769.642857,1,2.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,3.184553,1146.875000,3548.802689,20756.399245,23.250000,892.748355,624.923848,769.642857,1,2.0
C-B--C-3.3,3.184553,787.250000,1946.117604,11382.541522,12.750000,892.748355,624.923848,769.642857,1,2.0
C-B--C-4,3.184553,508.083333,1513.647025,8853.087850,9.916667,892.748355,624.923848,769.642857,1,2.0
C-B--C-4.9,3.184553,653.291667,2849.217930,16664.635953,18.666667,892.748355,624.923848,769.642857,1,2.0
C-W.6--C-7.3,3.068786,1674.291667,5138.042813,30051.619464,19.083333,1574.757352,1102.330146,1200.000000,2,3.0
C-X--C-5.8,3.068786,383.166667,1140.580285,6671.078065,9.833333,678.414718,474.890303,600.000000,1,3.0
C-X--C-6.5,3.068786,703.500000,2194.167159,12833.345090,18.916667,678.414718,474.890303,600.000000,1,3.0
C-C--C-2.5,3.184553,2590.666667,5915.282022,34597.571647,28.000000,1235.627559,864.939291,921.428571,2,4.0
C-C--C-3.6,3.184553,1058.041667,5527.971890,32332.254455,26.166667,1235.627559,864.939291,921.428571,2,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,1.825241,1146.875000,2034.012227,22790.411472,23.250000,980.232751,686.162926,769.642857,1,2.0
C-B--C-3.3,1.825241,787.250000,1115.426060,12497.967581,12.750000,980.232751,686.162926,769.642857,1,2.0
C-B--C-4,1.825241,508.083333,867.553602,9720.641452,9.916667,980.232751,686.162926,769.642857,1,2.0
C-B--C-4.9,1.825241,653.291667,1633.042074,18297.678028,18.666667,980.232751,686.162926,769.642857,1,2.0
C-W.6--C-7.3,1.758888,1674.291667,2944.892353,32996.511818,19.083333,1729.074855,1210.352399,1539.285714,2,2.0
C-X--C-5.8,1.758888,383.166667,653.728722,7324.806786,9.833333,744.895605,521.426924,600.000000,1,3.0
C-X--C-6.5,1.758888,703.500000,1257.596778,14090.941869,18.916667,744.895605,521.426924,600.000000,1,3.0
C-C--C-2.5,1.825241,2590.666667,3390.370503,37987.942149,28.000000,1356.712220,949.698554,1200.000000,2,3.0
C-C--C-3.6,1.825241,1058.041667,3168.381958,35500.636413,26.166667,1356.712220,949.698554,1200.000000,2,3.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [ ]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [ ]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [ ]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"] * rigid_ns["wall height"] / (1000 * rigid_ns["shear stiffness"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [ ]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [ ]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [ ]:
display_table(
    dataframe=rigid_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Roof",
    label="tab:wall_rigidities_ns_roof",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 5",
    label="tab:wall_rigidities_ns_level5",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,8090.047854,0.305045,26520.799552,0.108333
C-2.9,4686.795594,0.214800,21819.394625,0.089129
C-3,5693.000297,0.499297,11402.036347,0.046576
C-3.3,2098.466389,0.224196,9359.943935,0.038234
C-4,2820.021078,0.165297,17060.301482,0.069689
C-4.2,4686.795594,0.214800,21819.394625,0.089129
C-5,3543.156607,0.184473,19206.908116,0.078457
C-5.5,3334.782125,0.178947,18635.538127,0.076123
C-5.9,3534.223243,0.184236,19183.115698,0.078360


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,18531.738943,0.497356,37260.478276,0.105500
C-2.9,10735.965224,0.375209,28613.322884,0.081016
C-3,13040.861712,0.637397,20459.547895,0.057930
C-3.3,4806.922282,0.335245,14338.533755,0.040598
C-4,6459.775685,0.261815,24673.092579,0.069860
C-4.2,10735.965224,0.375209,28613.322884,0.081016
C-5,8116.250293,0.305740,26546.230114,0.075164
C-5.5,7638.930311,0.293083,26064.062639,0.073798
C-5.9,8095.786785,0.305198,26526.379327,0.075107


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,26340.790649,0.522443,50418.501132,0.122893
C-2.9,15259.971730,0.425529,35861.159828,0.087410
C-3,18536.123853,1.037954,17858.337054,0.043529
C-3.3,6832.501467,0.418295,16334.162302,0.039814
C-4,9181.847397,0.333997,27490.788686,0.067008
C-4.2,15259.971730,0.425529,35861.159828,0.087410
C-5,11536.340463,0.396433,29100.383259,0.070931
C-5.5,10857.883587,0.378442,28691.042165,0.069933
C-5.9,11507.253880,0.395661,29083.597882,0.070890


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,31773.174444,0.611521,51957.617658,0.116482
C-2.9,18407.106691,0.494620,37214.619543,0.083430
C-3,22358.914907,1.112013,20106.710203,0.045076
C-3.3,8241.600030,0.476069,17311.768899,0.038811
C-4,11075.462500,0.384211,28826.492220,0.064625
C-4.2,18407.106691,0.494620,37214.619543,0.083430
C-5,13915.533625,0.396014,35139.001740,0.078777
C-5.5,13097.155431,0.378048,34644.203088,0.077668
C-5.9,13880.448380,0.395244,35118.711814,0.078731


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,34886.769656,1.120298,31140.623979,0.088580
C-2.9,20210.901252,0.639721,31593.316822,0.089868
C-3,24549.964797,1.389699,17665.669454,0.050251
C-3.3,9049.231210,0.585219,15462.981521,0.043985
C-4,12160.796516,0.537682,22617.074337,0.064335
C-4.2,20210.901252,0.639721,31593.316822,0.089868
C-5,15279.178889,0.556060,27477.558428,0.078161
C-5.5,14380.604162,0.530202,27122.854204,0.077152
C-5.9,15240.655485,0.554952,27463.029842,0.078119


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [ ]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"] * rigid_ew["wall height"] / (1000 * rigid_ew["shear stiffness"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [ ]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [ ]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [ ]:
display_table(
    dataframe=rigid_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Roof",
    label="tab:wall_rigidities_ew_roof",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 5",
    label="tab:wall_rigidities_ew_level5",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,5284.969667,0.288776,18301.304125,0.110182
C-B--C-3.3,2898.209172,0.386193,7504.560029,0.045181
C-B--C-4,2254.162690,0.447825,5033.584270,0.030304
C-B--C-4.9,4243.129769,0.317821,13350.704819,0.080377
C-W.6--C-7.3,7651.707574,0.449859,17009.119781,0.102402
C-X--C-5.8,1698.581956,0.406186,4181.788331,0.025176
C-X--C-6.5,3267.611050,0.273353,11953.809726,0.071967
C-C--C-2.5,8809.192508,0.335904,26225.290548,0.157888
C-C--C-3.6,8232.400141,0.342907,24007.681536,0.144537


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,12106.192690,0.446855,27091.968628,0.104310
C-B--C-3.3,6638.879862,0.550001,12070.658738,0.046475
C-B--C-4,5163.573226,0.615257,8392.545306,0.032313
C-B--C-4.9,9719.667249,0.477608,20350.701841,0.078355
C-W.6--C-7.3,17527.640107,0.583392,30044.371503,0.115678
C-X--C-5.8,3890.913619,0.585896,6640.966937,0.025569
C-X--C-6.5,7485.062640,0.447062,16742.791981,0.064463
C-C--C-2.5,20179.071717,0.506210,39863.078905,0.153482
C-C--C-3.6,18857.822974,0.513772,36704.643994,0.141321


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,17207.596556,0.539460,31897.810198,0.107694
C-B--C-3.3,9436.423918,0.646890,14587.361525,0.049250
C-B--C-4,7339.440825,0.714856,10267.013205,0.034664
C-B--C-4.9,13815.418023,0.571491,24174.361253,0.081618
C-W.6--C-7.3,24913.576651,0.981614,25380.223717,0.085689
C-X--C-5.8,5530.497779,0.645857,8563.031570,0.028911
C-X--C-6.5,10639.177931,0.502535,21171.005813,0.071478
C-C--C-2.5,28682.289624,0.581563,49319.294270,0.166512
C-C--C-3.6,26804.282566,0.589545,45466.071510,0.153503


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,20756.399245,0.541629,38322.168845,0.126032
C-B--C-3.3,11382.541522,0.652040,17456.824917,0.057411
C-B--C-4,8853.087850,0.721891,12263.742254,0.040332
C-B--C-4.9,16664.635953,0.574548,29004.775528,0.095389
C-W.6--C-7.3,30051.619464,1.053304,28530.802261,0.093830
C-X--C-5.8,6671.078065,0.679894,9811.938156,0.032269
C-X--C-6.5,12833.345090,0.533450,24057.278494,0.079118
C-C--C-2.5,34597.571647,0.881258,39259.292910,0.129114
C-C--C-3.6,32332.254455,0.889531,36347.529099,0.119538


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,22790.411472,0.769911,29601.362418,0.121204
C-B--C-3.3,12497.967581,0.930134,13436.735630,0.055017
C-B--C-4,9720.641452,1.031500,9423.792075,0.038586
C-B--C-4.9,18297.678028,0.817682,22377.510872,0.091626
C-W.6--C-7.3,32996.511818,1.297903,25422.942689,0.104096
C-X--C-5.8,7324.806786,0.961240,7620.164632,0.031201
C-X--C-6.5,14090.941869,0.753817,18692.781964,0.076539
C-C--C-2.5,37987.942149,1.141285,33285.251648,0.136288
C-C--C-3.6,35500.636413,1.153707,30770.925503,0.125993


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines B-1 and B-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The top floor has a higher concentration of load on the N-E corner, but this area is relatively small and is only present on the top floor. We will assume the center of mass is the same as for a uniform floor weight.

All dimensions are measured from an origin at the S-E corner of the building (i.e., the intersection of C-9 and C-Z).

In [ ]:
%%render params
X_plan = fi(221, 6)  # ft
Y_plan = fi(115, 6)  # ft
CM_x = fi(102)  # ft
CM_y = fi(43, 3)  # ft

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [ ]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [ ]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [ ]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "C-2.1": {"x": fi(182, 8), "y": 0},
    "C-2.9": {"x": fi(160), "y": 0},
    "C-3": {"x": fi(145, 1), "y": 0},
    "C-3.3": {"x": fi(137, 1), "y": 0},
    "C-4": {"x": fi(118, 9), "y": 0},
    "C-4.2": {"x": fi(120, 4), "y": 0},
    "C-5": {"x": fi(102, 2), "y": 0},
    "C-5.5": {"x": fi(104, 11), "y": 0},
    "C-5.9": {"x": l_C59__C7 + l_C7__C83 + l_C83__C9, "y": 0},
    "C-6": {"x": l_C6__C7 + l_C7__C8 + l_C8__C9, "y": 0},
    "C-7": {"x": l_C7__C8 + l_C8__C9, "y": 0},
    "C-7.3": {"x": l_C73__C8 + l_C8__C9, "y": 0},
    "C-8--C-W": {"x": l_C8__C9, "y": 0},
    "C-8--C-W.6": {"x": l_C8__C9, "y": 0},
    "C-8.1": {"x": l_C81__C83 + l_C83__C9, "y": 0},
    "C-8.3": {"x": l_C83__C9, "y": 0},
    # E-W
    "C-B--C-2": {"x": 0, "y": fi(82, 4)},
    "C-B--C-4": {"x": 0, "y": fi(63, 4)},
    "C-B--C-4.9": {"x": 0, "y": fi(55, 8)},
    "C-X--C-5.8": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-X--C-6.5": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-W.6--C-7.3": {"x": 0, "y": l_CX__CY + l_CY__CZ + l_CW6__CX},
    "C-C--C-2.5": {"x": 0, "y": fi(69, 11)},
    "C-C--C-3.6": {"x": 0, "y": fi(56, 3)},
    "C-C--C-5.2": {"x": 0, "y": fi(47, 5)},
    "C-Y--C-5.8": {"x": 0, "y": l_CY__CZ},
    "C-Y--C-6.5": {"x": 0, "y": l_CY__CZ},
    "C-Y--C-7.2": {"x": 0, "y": l_CY__CZ},
}

In [ ]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
CMx = CM_x
CMxa_plus = CM_x__a__plus
CMxa_minus = CM_x__a__minus
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, abs(rigid["ex"]), rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
CMy = CM_y
CMya_plus = CM_y__a__plus
CMya_minus = CM_y__a__minus
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, abs(rigid["ey"]), rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ex"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ey"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [ ]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Roof",
    label="tab:center_of_rigidity_roof",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 5",
    label="tab:center_of_rigidity_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,103.100389,67.996223,8.666014,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,103.100389,48.746223,8.666014,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,103.100389,39.933723,8.666014,0.000000,0.000000,0.000000
D-4,93.104167,0.00,103.100389,9.996223,8.666014,0.000000,0.000000,0.000000
D-5,117.770833,0.00,103.100389,14.670444,12.319402,0.000000,0.000000,0.000000
D-6,136.354167,0.00,103.100389,33.253777,12.319402,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,103.100389,40.420444,12.319402,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,103.100389,59.337111,12.319402,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,103.100389,61.462111,12.319402,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,105.5439,70.439734,11.109525,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,105.5439,51.189734,11.109525,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,105.5439,42.377234,11.109525,0.000000,0.000000,0.000000
D-4,93.104167,0.00,105.5439,12.439734,11.109525,0.000000,0.000000,0.000000
D-5,117.770833,0.00,105.5439,12.226933,9.875891,0.000000,0.000000,0.000000
D-6,136.354167,0.00,105.5439,30.810266,9.875891,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,105.5439,37.976933,9.875891,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,105.5439,56.893600,9.875891,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,105.5439,59.018600,9.875891,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,105.526521,70.422354,11.092146,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,105.526521,51.172354,11.092146,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,105.526521,42.359854,11.092146,0.000000,0.000000,0.000000
D-4,93.104167,0.00,105.526521,12.422354,11.092146,0.000000,0.000000,0.000000
D-5,117.770833,0.00,105.526521,12.244312,9.893271,0.000000,0.000000,0.000000
D-6,136.354167,0.00,105.526521,30.827646,9.893271,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,105.526521,37.994312,9.893271,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,105.526521,56.910979,9.893271,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,105.526521,59.035979,9.893271,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,104.991038,69.886871,10.556663,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,104.991038,50.636871,10.556663,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,104.991038,41.824371,10.556663,0.000000,0.000000,0.000000
D-4,93.104167,0.00,104.991038,11.886871,10.556663,0.000000,0.000000,0.000000
D-5,117.770833,0.00,104.991038,12.779796,10.428754,0.000000,0.000000,0.000000
D-6,136.354167,0.00,104.991038,31.363129,10.428754,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,104.991038,38.529796,10.428754,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,104.991038,57.446462,10.428754,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,104.991038,59.571462,10.428754,0.000000,0.000000,0.000000


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [ ]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [ ]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Roof",
    label="tab:rigid_forces_roof",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 5",
    label="tab:rigid_forces_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,545266.336025,0.000000,67.996223,0.000000,0.176964,2674.850926,11134.569880,13809.420806,13033.541824
D-2.8,545266.336025,0.000000,48.746223,0.000000,0.107527,1165.165265,6765.582758,7930.748022,4309.638512
D-3.2,545266.336025,0.000000,39.933723,0.000000,0.083927,745.028657,5280.704064,6025.732721,5950.948502
D-4,545266.336025,0.000000,9.996223,0.000000,0.136996,304.421154,8619.807279,8924.228433,8385.718292
D-5,775137.809495,0.000000,14.670444,0.000000,0.127311,590.216162,8010.437820,8600.653982,6642.026392
D-6,775137.809495,0.000000,33.253777,0.000000,0.103416,1086.754728,6506.972449,7593.727177,3954.501262
D-6.2,775137.809495,0.000000,40.420444,0.000000,0.037182,474.934719,2339.490324,2814.425043,4005.692217
D-6.7,775137.809495,0.000000,59.337111,0.000000,0.033555,629.189831,2111.269696,2740.459527,3231.429025
D-6.8,775137.809495,0.000000,61.462111,0.000000,0.043825,851.201051,2757.484595,3608.685645,1689.301510


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,1.488811e+06,0.000000,70.439734,0.000000,0.177980,7300.469601,23851.478804,31151.948405,27759.851367
D-2.8,1.488811e+06,0.000000,51.189734,0.000000,0.096176,2866.904161,12888.801174,15755.705335,9179.003385
D-3.2,1.488811e+06,0.000000,42.377234,0.000000,0.082101,2026.015608,11002.525991,13028.541600,12674.793093
D-4,1.488811e+06,0.000000,12.439734,0.000000,0.129481,937.948360,17352.021139,18289.969499,17860.555214
D-5,1.323489e+06,0.000000,12.226933,0.000000,0.123405,781.074560,16537.747917,17318.822477,14146.704549
D-6,1.323489e+06,0.000000,30.810266,0.000000,0.093891,1497.479070,12582.493598,14079.972667,8422.604443
D-6.2,1.323489e+06,0.000000,37.976933,0.000000,0.040584,797.846009,5438.768814,6236.614823,8531.634921
D-6.7,1.323489e+06,0.000000,56.893600,0.000000,0.035228,1037.501108,4720.919194,5758.420302,6882.548938
D-6.8,1.323489e+06,0.000000,59.018600,0.000000,0.046430,1418.494342,6222.144853,7640.639195,3598.005781


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,2.036708e+06,0.000000,70.422354,0.000000,0.138709,8646.376536,25469.422508,34115.799044,38035.266225
D-2.8,2.036708e+06,0.000000,51.172354,0.000000,0.100113,4534.637775,18382.420512,22917.058288,12576.646497
D-3.2,2.036708e+06,0.000000,42.359854,0.000000,0.106268,3984.515908,19512.665544,23497.181452,17366.416098
D-4,2.036708e+06,0.000000,12.422354,0.000000,0.144178,1585.328576,26473.469893,28058.798469,24471.707845
D-5,1.816574e+06,0.000000,12.244312,0.000000,0.138435,1338.208367,25419.120710,26757.329077,19383.161193
D-6,1.816574e+06,0.000000,30.827646,0.000000,0.098230,2390.697940,18036.635567,20427.333507,11540.263601
D-6.2,1.816574e+06,0.000000,37.994312,0.000000,0.039944,1198.157770,7334.433943,8532.591713,11689.652126
D-6.7,1.816574e+06,0.000000,56.910979,0.000000,0.045391,2039.444215,8334.642102,10374.086317,9430.150677
D-6.8,1.816574e+06,0.000000,59.035979,0.000000,0.051530,2401.681492,9461.714146,11863.395637,4929.821344


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,2.238282e+06,0.000000,69.886871,0.000000,0.139487,9496.022815,29574.904001,39070.926817,43919.898521
D-2.8,2.238282e+06,0.000000,50.636871,0.000000,0.108222,5338.173373,22945.789422,28283.962795,14522.444371
D-3.2,2.238282e+06,0.000000,41.824371,0.000000,0.097845,3986.387041,20745.658966,24732.046007,20053.263942
D-4,2.238282e+06,0.000000,11.886871,0.000000,0.154613,1790.292159,32781.856697,34572.148857,28257.852038
D-5,2.211162e+06,0.000000,12.779796,0.000000,0.128049,1574.764790,27149.585964,28724.350754,22382.030077
D-6,2.211162e+06,0.000000,31.363129,0.000000,0.106145,3203.593657,22505.541333,25709.134991,13325.717329
D-6.2,2.211162e+06,0.000000,38.529796,0.000000,0.036487,1352.870884,7736.260173,9089.131057,13498.218524
D-6.7,2.211162e+06,0.000000,57.446462,0.000000,0.041820,2311.857773,8866.847629,11178.705402,10889.137947
D-6.8,2.211162e+06,0.000000,59.571462,0.000000,0.049284,2825.263373,10449.417954,13274.681327,5692.539441


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [ ]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    caption="Final shear wall design values - Roof",
    label="tab:shear_walls_roof",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    caption="Final shear wall design values - Level 5",
    label="tab:shear_walls_level5",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,13809.420806,29.000000,476.186924,333.330847,460.714286,1,4.0
D-2.8,7930.748022,29.000000,273.474070,191.431849,310.714286,1,6.0
D-3.2,6025.732721,20.000000,301.286636,210.900645,310.714286,1,6.0
D-4,8924.228433,29.000000,307.732015,215.412410,310.714286,1,6.0
D-5,8600.653982,29.000000,296.574275,207.601993,310.714286,1,6.0
D-6,7593.727177,29.000000,261.852661,183.296863,310.714286,1,6.0
D-6.2,4005.692217,10.583333,378.490603,264.943422,310.714286,1,6.0
D-6.7,3231.429025,10.583333,305.331876,213.732313,310.714286,1,6.0
D-6.8,3608.685645,20.000000,180.434282,126.303998,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,31151.948405,29.000000,1074.205117,751.943582,769.642857,1,2.0
D-2.8,15755.705335,29.000000,543.300184,380.310129,460.714286,1,4.0
D-3.2,13028.541600,20.000000,651.427080,455.998956,460.714286,1,4.0
D-4,18289.969499,29.000000,630.688603,441.482022,460.714286,1,4.0
D-5,17318.822477,29.000000,597.200775,418.040543,460.714286,1,4.0
D-6,14079.972667,29.000000,485.516299,339.861409,460.714286,1,4.0
D-6.2,8531.634921,10.583333,806.138733,564.297113,600.000000,1,3.0
D-6.7,6882.548938,10.583333,650.319585,455.223709,460.714286,1,4.0
D-6.8,7640.639195,20.000000,382.031960,267.422372,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,38035.266225,29.000000,1311.560904,918.092633,921.428571,2,4.0
D-2.8,22917.058288,29.000000,790.243389,553.170372,600.000000,1,3.0
D-3.2,23497.181452,20.000000,1174.859073,822.401351,921.428571,2,4.0
D-4,28058.798469,29.000000,967.544775,677.281342,769.642857,1,2.0
D-5,26757.329077,29.000000,922.666520,645.866564,769.642857,1,2.0
D-6,20427.333507,29.000000,704.390811,493.073567,600.000000,1,3.0
D-6.2,11689.652126,10.583333,1104.534059,773.173841,921.428571,2,4.0
D-6.7,10374.086317,10.583333,980.228628,686.160040,769.642857,1,2.0
D-6.8,11863.395637,20.000000,593.169782,415.218847,460.714286,1,4.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,43919.898521,29.000000,1514.479259,1060.135482,1200.000000,2,3.0
D-2.8,28283.962795,29.000000,975.309062,682.716343,769.642857,1,2.0
D-3.2,24732.046007,20.000000,1236.602300,865.621610,921.428571,2,4.0
D-4,34572.148857,29.000000,1192.143064,834.500145,921.428571,2,4.0
D-5,28724.350754,29.000000,990.494854,693.346398,769.642857,1,2.0
D-6,25709.134991,29.000000,886.521896,620.565327,769.642857,1,2.0
D-6.2,13498.218524,10.583333,1275.422223,892.795556,921.428571,2,4.0
D-6.7,11178.705402,10.583333,1056.255629,739.378940,769.642857,1,2.0
D-6.8,13274.681327,20.000000,663.734066,464.613846,600.000000,1,3.0


### Shear Wall Holddown Forces

The shear forces get converted to overturning forces in the end posts of the shear walls. This overturning moment is counteracted by the dead load tributary to each shear wall according to ASCE 7-16 Sec. 2.4.5. The applicable equation is:

$$
0.6 D + 0.7 E_v + 0.7 E_h \quad\text{(ASCE 7-16 Eq. 2.4.5-8)}
$$

There are no vertical seismic effects, $E_v$, so we will only consider the horizontal seismic effects, $E_h$, and dead loads, $D$.

The overturning moment is:

$$
M_{OT} = V h_{wall}
$$

Using the area dead loads from earlier, along with the tributary areas for each shear wall, we can get the dead load that is applied to each wall. This resists the overturning moment at the centroid of the wall.

$$
M_R = \frac{D l_{wall}}{2}
$$

We will assume that the holddowns are 6" inset from the end posts. The tension caused by horizontal load effects counteracted by dead loads is:

$$
T_{seismic} = \frac{M_{OT} - M_R}{l_{wall} - 0.5}\quad\text{units in feet}
$$

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [ ]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [ ]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,376.353229,376.353229,62.920082,62.920082,50.972900,101.945800,62.920082
Level 4,548.489146,924.842375,134.012087,79.477516,74.286814,148.573628,79.477516
Level 3,550.150271,1474.992646,183.617171,68.486468,74.511795,149.023590,74.511795
Level 2,560.117021,2035.109667,212.025532,58.355140,75.861682,151.723363,75.861682


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [ ]:
# Set up diaphragm dataframe
levels = ["Roof", "Level 5", "Level 4", "Level 3", "Level 2"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [ ]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 35
diaphragms_ns["L_cant"] = 35
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [ ]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.timber.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [55]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 35
diaphragms_ew["depth"] = X_plan
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]


KeyError: 'F_px'

In [ ]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [ ]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [ ]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Roof    N-S          62920.082384   4.584265       4607.185937   
        E-W          62920.082384   4.584265       2582.540695   
Level 4 N-S          79477.516242   5.790615       5819.567954   
        E-W          79477.516242   5.790615       3262.136861   
Level 3 N-S          74511.795196   5.428820       5455.963850   
        E-W          74511.795196   5.428820       3058.319952   
Level 2 N-S          75861.681716   5.527170       5554.806350   
        E-W          75861.681716   5.527170       3113.725742   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Roof    N-S            10750.100519         68.763969        160.449261  
        E-W            29112.276924         12.606728        142.112203  
Level 4 N-S            13578.991893         86.859223        202.671521  
        E-W            36773.179157         15.924191        179.509061  
Level 3 N-S            12730.582318         81.432296        190.008691  
        E-W            34475.606733         14.929254        168.293412  
Level 2 N-S            12961.214816         82.907557        193.450967  
        E-W            35100.181093         15.199719        171.342285

In [ ]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Roof    N-S               68.763969            189.285714          0.363281   
        E-W               12.606728            189.285714          0.066602   
Level 4 N-S               86.859223            189.285714          0.458879   
        E-W               15.924191            189.285714          0.084128   
Level 3 N-S               81.432296            189.285714          0.430208   
        E-W               14.929254            189.285714          0.078872   
Level 2 N-S               82.907557            189.285714          0.438002   
        E-W               15.199719            189.285714          0.080300   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Roof    N-S              160.449261            189.285714          0.847656  
        E-W              142.112203            189.285714          0.750781  
Level 4 N-S              202.671521            214.285714          0.945800  
        E-W              179.509061            189.285714          0.948350  
Level 3 N-S              190.008691            214.285714          0.886707  
        E-W              168.293412            189.285714          0.889097  
Level 2 N-S              193.450967            214.285714          0.902771  
        E-W              171.342285            189.285714          0.905205

In [ ]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'